In [1]:
import json

In [8]:
data = json.load(open("/Users/mc03002/Downloads/trajectory_data_0_1000.json"))

In [9]:
len(data)

1000

In [22]:
x = data[2]
x.keys()

dict_keys(['idx', 'question', 'prompt_ids', 'trajectory', 'final_output', 'generated_text', 'llm_answer', 'gt_answer', 'is_correct', 'nfe'])

In [35]:
print(x['question']) 
print("----------")
print(x["generated_text"])
x["gt_answer"], x["llm_answer"]

Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
        This is an example for a response to the question:
        In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5
        Now answer with a response of your own, including the thinking process
        please using this format `thinking #### answer`
        and dont repeat question:
----------
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more mon

('5', '5')

In [36]:
generation_list = list(map(lambda x: x["generated_text"], data))
gt_answer_list = list(map(lambda x: x["gt_answer"], data))
llm_answer_list = list(map(lambda x: x["llm_answer"], data))

In [46]:
question_list = list(map(lambda x: x.split('\n')[3], generation_list))
gt_cot_list = list(map(lambda x: x.split("Now answer with a response of your own")[0].split("This is an example for a response to the question:")[-1].strip().strip("\n"), generation_list))
distill_cot_list = list(map(lambda x: x.split('dont repeat question:<|im_end|>\n<|im_start|>assistant\n')[-1], generation_list))

In [47]:
import pandas as pd
df = pd.DataFrame(
    {
        "question": question_list,
        "gt_cot": gt_cot_list,
        "distill_cot": distill_cot_list,
        "gt_answer": gt_answer_list,
        "llm_answer": llm_answer_list
    }
)
df

,question,gt_cot,distill_cot,gt_answer,llm_answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...,Natalia sold clips to 48 of her friends in Apr...,72,72
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...,Weng earns $12 an hour for babysitting. Yester...,10,10
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<...","Betty has only half of the money she needs, wh...",5,5
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....,Maila read 12 x 2 = <<12*2=24>>24 pages today....,42,42
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,624,624
...,...,...,...,...,...
995,"Josh and Anna were both born on August 17th, b...",We know that Josh must be 30 years older than ...,We know that Josh must be 30 years older than ...,28,28
996,A club opens up and charges $20 to enter. Jam...,He buys 2*5=<<2*5=10>>10 drinks for his friend...,He buys 2*5=<<2*5=10>>10 drinks for his friend...,163,163
997,Ittymangnark and Kingnook are an Eskimo couple...,If the dog eats two eyes and Oomyapeck eats 22...,If Oomyapeck eats 22 eyes and the dog eats 2 e...,4,4
998,Lydia has a small pool she uses to bathe her d...,With a garden hose that fills at a rate of 1.6...,To find the time it takes for Lydia to fill th...,40,40


In [48]:
df["distill_correct"] = (df["gt_answer"] == df["llm_answer"]).astype(int)
df["distill_correct"].mean()

np.float64(0.914)

In [49]:
df.to_excel("distill_v1_0_1000.xlsx")